# Normalization strategy selection on real data

This study runs the exhaustive grid of every `ChannelStrategy` preset against every normalized data kind
in the dataset pipeline — all input slot kinds and all output roles — on real preprocessed data, and ranks
the strategies per slot with a transparent metric battery. The configured standards in
`configuration/norm_config.py` are evaluated head-to-head against every alternative.

```
complex pass / ifg stacks ──► per-slot value pools ──► fit all 7 presets ──► normalize
      └─► DEM, GT parameters ──┘                              └─► metric battery ──► composite ranking vs configured standard
```

| Axis | Members |
|------|---------|
| Strategies (7) | MIN_MAX, MIN_MAX_LOG1P, ROBUST_IQR, ROBUST_IQR_LOG1P, ZSCORE, ZSCORE_LOG1P, FIXED_DIV_PI |
| Input slot kinds (9) | pass/mag, pass/raw_re_im, pass/norm_re_im, pass/phase, ifg/mag, ifg/raw_re_im, ifg/norm_re_im, ifg/phase, dem/elevation |
| Output roles (3) | out/amp, out/mu, out/sigma |

**84 strategy-slot combinations in total.**

**Data requirement.** Unlike the confirmation suites in the sibling directories, this notebook needs a real
preprocessing run directory and a parameters file. Edit `dataset_path` and `parameters_path` in the
configuration cell before running; the defaults point at the server data mount.

Modules exercised:

- `configuration.norm_config.ChannelStrategy`, `NormMethod`, `Presets`
- `pipelines.dataset_pipeline.spatial.Layout`, `Cropper`
- `pipelines.dataset_pipeline.normalization.Normalizer` (inverse-transform ceiling)
- `tools.regions.SplitRegions`, `CropRegion`

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import matplotlib.pyplot as plt

import matplotlib
matplotlib.rcParams.update({
    "figure.dpi"       : 130,
    "font.size"        : 10,
    "axes.labelsize"   : 11,
    "axes.titlesize"   : 12,
    "legend.fontsize"  : 8,
    "xtick.labelsize"  : 9,
    "ytick.labelsize"  : 9,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
})

SEED = 42
np.random.seed(SEED)

print("repo root on path:", REPO_ROOT)

## Study configuration

| Field | Meaning |
|-------|---------|
| `dataset_path` | Preprocessing run directory containing `data/dataset.json` and the artifact `.npy` files. **Edit before running.** |
| `parameters_path` | Ground-truth Gaussian parameter cube `(3·N_g, az, rg)`. **Edit before running.** |
| `secondary_labels` | Secondary pass selection, matching the training preset. |
| `train_ratio`, `val_ratio` | Azimuth split ratios; pools are fitted on the **train** region only, the same leakage rule as `DatasetPipeline.run`. |
| `max_azimuth_lines` | Clips the train region to bound memory; set `None` to use the full train split. |
| `max_values_per_pool` | Seeded random subsample cap per slot pool. |
| `max_plot_values` | Seeded subsample cap applied inside the histogram cells only, so plotting memory stays bounded regardless of pool size. Scoring still uses the full pool. |
| `amp_threshold` | Active-pixel gate for output pools, matching `StatsComputer.compute_output_stats`. |
| `outlier_limit` | Distance from the median (in normalized units) beyond which a value counts as an outlier. |
| `roundtrip_tolerance` | Relative reconstruction error above which a strategy is disqualified as non-invertible on that slot. |
| `score_weights` | Weights of the composite score; they sum to 1 and are stated again in the metric battery section. |

In [ ]:
from dataclasses import dataclass, field
from typing      import Optional


def _default_score_weights() -> dict:
    return {
        "scale"    : 0.20,
        "center"   : 0.10,
        "skewness" : 0.15,
        "kurtosis" : 0.10,
        "tails"    : 0.15,
        "outliers" : 0.15,
        "entropy"  : 0.15,
    }


@dataclass
class StudyConfig:
    dataset_path        : Path          = Path("/ste/rnd/User/vice_vi/Dataset/base_dataset_w20_10")
    parameters_path     : Path          = Path("/ste/rnd/User/vice_vi/Dataset/base_dataset_w20_10/params/params_Ng3_sigonly_k5/parameters_Ng3_sigonly_k5.npy")
    secondary_labels    : tuple         = ("FL01_PS04", "FL01_PS06", "FL01_PS08", "FL01_PS26")

    train_ratio         : float         = 0.70
    val_ratio           : float         = 0.15
    max_azimuth_lines   : Optional[int] = 2048

    max_values_per_pool : int           = 2_000_000
    amp_threshold       : float         = 1e-2

    histogram_bins      : int           = 64
    max_plot_values     : int           = 300_000
    outlier_limit       : float         = 6.0
    roundtrip_tolerance : float         = 1e-3
    score_weights       : dict          = field(default_factory=_default_score_weights)


study_config = StudyConfig()

print("dataset path    :", study_config.dataset_path)
print("parameters path :", study_config.parameters_path)
print("dataset exists  :", study_config.dataset_path.exists())
print("params exist    :", study_config.parameters_path.exists())

## Strategy catalogue

Every preset is the same affine map applied after an optional log transform:

$$y = \frac{T(x) - \mathrm{loc}}{\mathrm{scale}},
\qquad
T(x) = \begin{cases} \log\!\bigl(1 + \max(x, 0)\bigr) & \text{log1p variants} \\ x & \text{otherwise} \end{cases}$$

with the inverse $\hat{x} = T^{-1}(y \cdot \mathrm{scale} + \mathrm{loc})$, where the `expm1` input is
clamped at `Normalizer.EXPM1_INPUT_CEIL` exactly as in the production code path. The fitting rules differ
only in how `loc` and `scale` are estimated from the (transformed) training pool:

| Method | $\mathrm{loc}$ | $\mathrm{scale}$ |
|--------|----------------|------------------|
| `MIN_MAX_P999` | $P_{0.1}$ | $P_{99.9} - P_{0.1}$ |
| `ROBUST_IQR`   | $P_{50}$  | $P_{75} - P_{25}$ |
| `ZSCORE`       | $\mu$     | $\sigma$ |
| `FIXED_DIV_PI` | $0$       | $\pi$ |

The log1p pre-transform clamps negatives to zero before the logarithm, so it is only invertible on
non-negative data; the round-trip gate below disqualifies it automatically on signed slots.

> **What you should see:** the rendered table of configured standards must match the slot mapping in
> `configuration/norm_config.py` — magnitudes and amplitudes on `MIN_MAX_LOG1P`, raw re/im and mu on
> `MIN_MAX`, normalized re/im on `ROBUST_IQR`, phases on `FIXED_DIV_PI`, sigma on `MIN_MAX_LOG1P`,
> DEM on `ZSCORE`.

In [ ]:
from IPython.display import Markdown, display

from configuration.norm_config                import ChannelStrategy, NormMethod, Presets
from pipelines.dataset_pipeline.normalization import Normalizer


class MarkdownRenderer:
    @staticmethod
    def table(headers: list[str], rows: list[tuple]) -> None:
        head = "| " + " | ".join(headers) + " |"
        rule = "|" + "|".join(["---"] * len(headers)) + "|"
        body = "\n".join("| " + " | ".join(str(value) for value in row) + " |" for row in rows)

        display(Markdown("\n".join([head, rule, body])))

    @staticmethod
    def heading(text: str, level: int = 3) -> None:
        display(Markdown("#" * level + " " + text))


class StrategyCatalog:
    PRESETS = {
        "MIN_MAX"          : Presets.MIN_MAX,
        "MIN_MAX_LOG1P"    : Presets.MIN_MAX_LOG1P,
        "ROBUST_IQR"       : Presets.ROBUST_IQR,
        "ROBUST_IQR_LOG1P" : Presets.ROBUST_IQR_LOG1P,
        "ZSCORE"           : Presets.ZSCORE,
        "ZSCORE_LOG1P"     : Presets.ZSCORE_LOG1P,
        "FIXED_DIV_PI"     : Presets.FIXED_DIV_PI,
    }

    @classmethod
    def name_of(cls, strategy: ChannelStrategy) -> str:
        for name, preset in cls.PRESETS.items():
            if preset.norm_method is strategy.norm_method and preset.apply_log1p == strategy.apply_log1p:
                return name

        raise ValueError(f"No preset matches {strategy}")

    @staticmethod
    def standard_for(slot: str) -> ChannelStrategy:
        return ChannelStrategy.from_slot(slot)


INPUT_SLOTS  = ["pass/mag", "pass/raw_re_im", "pass/norm_re_im", "pass/phase", "ifg/mag", "ifg/raw_re_im", "ifg/norm_re_im", "ifg/phase", "dem/elevation"]
OUTPUT_SLOTS = ["out/amp", "out/mu", "out/sigma"]
ALL_SLOTS    = INPUT_SLOTS + OUTPUT_SLOTS

standard_rows = []
for slot in ALL_SLOTS:
    standard = StrategyCatalog.standard_for(slot)
    standard_rows.append((slot, standard.norm_method.value, standard.apply_log1p, f"`{StrategyCatalog.name_of(standard)}`"))

MarkdownRenderer.heading("Configured standards (current `_SLOT_STRATEGIES`)")
MarkdownRenderer.table(["Slot", "Method", "log1p", "Preset"], standard_rows)

## Data loading and slot pools

`Layout` reads `data/dataset.json` from the preprocessing run; `Cropper.load_split` returns the complex
pass stack (1 primary + the selected secondaries), the complex interferograms, the float DEM, and the
`(3·N_g, az, rg)` ground-truth parameter cube for the requested crop.

Pools follow the production grouping exactly:

- every channel of a slot kind is pooled together, matching `StatsComputer._collect` grouping;
- statistics are fitted on the **train** region only, the same leakage rule as `DatasetPipeline.run`;
- output roles pool **active pixels only** (amplitude above `amp_threshold`), matching
  `StatsComputer.compute_output_stats`;
- each pool is subsampled to `max_values_per_pool` with a seeded generator.

> **What you should see:** twelve pools, each capped at the configured maximum. Phases bounded by
> $\pm\pi$, normalized re/im inside $[-1, 1]$, magnitudes and amplitudes non-negative and strongly
> right-skewed, mu within the elevation axis span, sigma positive, DEM in metres. Any NaN count above
> zero warrants investigation of the preprocessing run.

In [ ]:
from tools.logger                       import Logger
from tools.regions                      import CropRegion, SplitRegions
from pipelines.dataset_pipeline.spatial import Cropper, Layout


class PoolBuilder:
    def __init__(self, config: StudyConfig) -> None:
        self.config = config
        self.logger = Logger(log_dir=str(REPO_ROOT / "logs" / "notebooks"), name="normalization_strategy_study", level="INFO")
        self.rng    = np.random.default_rng(SEED)

    def _train_region(self, splits: SplitRegions) -> CropRegion:
        region = splits.regions("train")[0]

        if self.config.max_azimuth_lines is None:
            return region

        azimuth_end = min(region.azimuth_end, region.azimuth_start + self.config.max_azimuth_lines)

        return CropRegion(region.azimuth_start, azimuth_end, region.range_start, region.range_end)

    def _subsample(self, values: np.ndarray) -> np.ndarray:
        flat = np.asarray(values).ravel()
        flat = flat[np.isfinite(flat)]

        if len(flat) > self.config.max_values_per_pool:
            indices = self.rng.choice(len(flat), self.config.max_values_per_pool, replace=False)
            flat    = flat[indices]

        return flat.astype(np.float64)

    def _complex_pools(self, prefix: str, stack: np.ndarray) -> dict:
        magnitude = np.abs(stack)
        safe_mag  = np.where(magnitude > 0, magnitude, 1.0)

        return {
            f"{prefix}/mag"        : self._subsample(magnitude),
            f"{prefix}/raw_re_im"  : self._subsample(np.concatenate([stack.real.ravel(), stack.imag.ravel()])),
            f"{prefix}/norm_re_im" : self._subsample(np.concatenate([(stack.real / safe_mag).ravel(), (stack.imag / safe_mag).ravel()])),
            f"{prefix}/phase"      : self._subsample(np.angle(stack)),
        }

    def _output_pools(self, parameters: np.ndarray) -> dict:
        n_gaussians = parameters.shape[0] // 3

        amplitude_values = []
        mu_values        = []
        sigma_values     = []

        for gaussian in range(n_gaussians):
            amplitude = parameters[gaussian * 3 + 0].ravel().astype(np.float64)
            mu        = parameters[gaussian * 3 + 1].ravel().astype(np.float64)
            sigma     = parameters[gaussian * 3 + 2].ravel().astype(np.float64)
            active    = amplitude > self.config.amp_threshold

            amplitude_values.append(amplitude[active])
            mu_values.append(mu[active])
            sigma_values.append(sigma[active])

        return {
            "out/amp"   : self._subsample(np.concatenate(amplitude_values)),
            "out/mu"    : self._subsample(np.concatenate(mu_values)),
            "out/sigma" : self._subsample(np.concatenate(sigma_values)),
        }

    def build(self) -> dict:
        layout  = Layout(self.config.dataset_path, logger=self.logger, parameters_path=self.config.parameters_path)
        splits  = SplitRegions.from_ratios(layout.global_crop, self.config.train_ratio, self.config.val_ratio)
        cropper = Cropper(layout, splits, logger=self.logger, secondary_labels=self.config.secondary_labels)
        region  = self._train_region(splits)
        arrays  = cropper.load_split(region)

        n_secondaries  = arrays["n_secondaries"]
        passes         = arrays["inputs"][:1 + n_secondaries]
        interferograms = arrays["inputs"][1 + n_secondaries:]

        pools = {}
        pools.update(self._complex_pools("pass", passes))
        pools.update(self._complex_pools("ifg", interferograms))
        pools["dem/elevation"] = self._subsample(arrays["dem"])
        pools.update(self._output_pools(arrays["parameters"]))

        return pools


pools = PoolBuilder(study_config).build()

In [ ]:
class PoolSummary:
    @staticmethod
    def moments(values: np.ndarray) -> tuple:
        mean     = float(values.mean())
        std      = float(values.std())
        centered = values - mean
        safe_std = max(std, 1e-12)
        skewness = float((centered ** 3).mean() / safe_std ** 3)

        return mean, std, skewness

    @classmethod
    def render(cls, pools: dict) -> None:
        rows = []
        for slot in ALL_SLOTS:
            values               = pools[slot]
            mean, std, skewness  = cls.moments(values)
            negative_percent     = 100.0 * float((values < 0).mean())

            rows.append((
                f"`{slot}`",
                f"{len(values):,}",
                f"{values.min():.4g}",
                f"{values.max():.4g}",
                f"{mean:.4g}",
                f"{std:.4g}",
                f"{skewness:+.2f}",
                f"{negative_percent:.1f}%",
            ))

        MarkdownRenderer.heading("Slot pools — raw statistics")
        MarkdownRenderer.table(["Slot", "Values", "Min", "Max", "Mean", "Std", "Skewness", "Negative"], rows)


PoolSummary.render(pools)

## Raw distributions

One histogram per slot, clipped to the central 99.9% so extreme tails do not flatten the plot; the count
axis is logarithmic so both the body and the tail of each distribution stay visible. These are the
distributions every strategy below has to tame.

In [ ]:
class PlotSampler:
    def __init__(self, max_values: int) -> None:
        self.max_values = max_values
        self.rng        = np.random.default_rng(SEED)

    def take(self, values: np.ndarray) -> np.ndarray:
        if len(values) <= self.max_values:
            return values

        indices = self.rng.integers(0, len(values), self.max_values)

        return values[indices]


class RawDistributionPlotter:
    def __init__(self, sampler: PlotSampler) -> None:
        self.sampler = sampler

    def plot(self, slot: str, values: np.ndarray) -> None:
        sample    = self.sampler.take(values)
        low, high = np.percentile(sample, [0.05, 99.95])

        figure, axis = plt.subplots(figsize=(7.0, 3.0))
        axis.hist(np.clip(sample, low, high), bins=120, color="#3a6ea5", alpha=0.9)
        axis.set_yscale("log")
        axis.set_title(f"Raw distribution — {slot}")
        axis.set_xlabel("value")
        axis.set_ylabel("count (log)")
        plt.show()
        plt.close(figure)


plot_sampler = PlotSampler(study_config.max_plot_values)
raw_plotter  = RawDistributionPlotter(plot_sampler)
for slot in ALL_SLOTS:
    raw_plotter.plot(slot, pools[slot])

## Metric battery

Each strategy-slot pair is scored on the **normalized** pool $y$. Every metric maps to a sub-score in
$[0, 1]$ and the composite is their weighted mean:

| Sub-score | Measures | Definition | Full credit | Weight |
|-----------|----------|------------|-------------|--------|
| scale | spread health | $e^{-\lvert \ln \sigma_y - \ln \mathrm{clip}(\sigma_y,\,0.25,\,1.5) \rvert}$ | $\sigma_y \in [0.25, 1.5]$ | 0.20 |
| center | offset from zero | $e^{-\lvert \bar{y} \rvert}$ | $\bar{y} = 0$ | 0.10 |
| skewness | asymmetry | $e^{-\lvert \tilde{\mu}_3 \rvert / 2}$ | $\tilde{\mu}_3 = 0$ | 0.15 |
| kurtosis | tail mass vs. Gaussian | $e^{-\lvert \kappa \rvert / 10}$ | excess $\kappa = 0$ | 0.10 |
| tails | spread of extremes | $e^{-\max(0,\, (P_{99.9} - P_{0.1}) - 8) / 8}$ | width $\le 8$ | 0.15 |
| outliers | mass far from the median | $e^{-200 f_{\mathrm{out}}}$, $f_{\mathrm{out}} = \Pr\bigl(\lvert y - \mathrm{med} \rvert > 6\bigr)$ | $f_{\mathrm{out}} = 0$ | 0.15 |
| entropy | use of the value range | $H_{64\text{-bin}} / \ln 64$ over $[P_{0.1}, P_{99.9}]$ | uniform occupancy | 0.15 |

Two hard gates zero the composite regardless of the sub-scores:

1. **Finiteness** — any non-finite normalized value disqualifies the pair.
2. **Invertibility** — normalize-then-denormalize must reconstruct the pool with relative error below
   `roundtrip_tolerance` (compared at the 99.9th percentile of absolute error against the 99.9th
   percentile of absolute value). This is what eliminates log1p variants on signed slots.

The composite measures **distribution quality for network consumption** — it does not measure downstream
task accuracy. Treat it as a strong prior; the definitive selection for a contested slot is a training
sweep over the contenders.

In [ ]:
class NormalizationMetrics:
    def __init__(self, config: StudyConfig) -> None:
        self.config = config

    def _moments(self, values: np.ndarray) -> tuple:
        mean     = float(values.mean())
        std      = float(values.std())
        centered = values - mean
        safe_std = max(std, 1e-12)
        skewness = float((centered ** 3).mean() / safe_std ** 3)
        kurtosis = float((centered ** 4).mean() / safe_std ** 4 - 3.0)

        return mean, std, skewness, kurtosis

    def _entropy(self, values: np.ndarray) -> float:
        low, high = np.percentile(values, [0.1, 99.9])
        if high <= low:
            return 0.0

        counts, _     = np.histogram(values, bins=self.config.histogram_bins, range=(low, high))
        probabilities = counts / max(counts.sum(), 1)
        nonzero       = probabilities[probabilities > 0]

        return float(-(nonzero * np.log(nonzero)).sum() / np.log(self.config.histogram_bins))

    def compute(self, normalized: np.ndarray) -> dict:
        mean, std, skewness, kurtosis = self._moments(normalized)
        p_low, median, p_high         = np.percentile(normalized, [0.1, 50.0, 99.9])

        return {
            "mean"             : mean,
            "std"              : std,
            "skewness"         : skewness,
            "kurtosis"         : kurtosis,
            "tail_width"       : float(p_high - p_low),
            "outlier_fraction" : float((np.abs(normalized - median) > self.config.outlier_limit).mean()),
            "entropy"          : self._entropy(normalized),
            "finite_fraction"  : float(np.isfinite(normalized).mean()),
        }


class SubScores:
    def __init__(self, config: StudyConfig) -> None:
        self.config = config

    def compute(self, metrics: dict) -> tuple:
        std_clipped = min(max(metrics["std"], 0.25), 1.5)

        scores = {
            "scale"    : float(np.exp(-abs(np.log(max(metrics["std"], 1e-12)) - np.log(std_clipped)))),
            "center"   : float(np.exp(-abs(metrics["mean"]))),
            "skewness" : float(np.exp(-abs(metrics["skewness"]) / 2.0)),
            "kurtosis" : float(np.exp(-abs(metrics["kurtosis"]) / 10.0)),
            "tails"    : float(np.exp(-max(0.0, metrics["tail_width"] - 8.0) / 8.0)),
            "outliers" : float(np.exp(-metrics["outlier_fraction"] * 200.0)),
            "entropy"  : metrics["entropy"],
        }

        weights   = self.config.score_weights
        composite = sum(weights[key] * scores[key] for key in weights) / sum(weights.values())

        return scores, composite


class StrategyEvaluator:
    def __init__(self, config: StudyConfig) -> None:
        self.config  = config
        self.metrics = NormalizationMetrics(config)
        self.scoring = SubScores(config)

    def transform(self, values: np.ndarray, strategy: ChannelStrategy, loc: float, scale: float) -> np.ndarray:
        data = np.log1p(np.maximum(values, 0.0)) if strategy.apply_log1p else values

        return (data - loc) / scale

    def inverse_transform(self, normalized: np.ndarray, strategy: ChannelStrategy, loc: float, scale: float) -> np.ndarray:
        data = normalized * scale + loc

        if strategy.apply_log1p:
            return np.expm1(np.minimum(data, Normalizer.EXPM1_INPUT_CEIL))

        return data

    def _roundtrip_error(self, values: np.ndarray, reconstructed: np.ndarray) -> float:
        reference = float(np.percentile(np.abs(values), 99.9)) + 1e-12

        return float(np.percentile(np.abs(reconstructed - values), 99.9)) / reference

    def evaluate(self, slot: str, preset_name: str, strategy: ChannelStrategy, values: np.ndarray) -> dict:
        loc, scale    = strategy.fit(values)
        normalized    = self.transform(values, strategy, loc, scale)
        reconstructed = self.inverse_transform(normalized, strategy, loc, scale)

        metrics    = self.metrics.compute(normalized)
        roundtrip  = self._roundtrip_error(values, reconstructed)
        invertible = roundtrip <= self.config.roundtrip_tolerance
        valid      = invertible and metrics["finite_fraction"] == 1.0

        scores, composite = self.scoring.compute(metrics)

        return {
            "slot"            : slot,
            "strategy"        : preset_name,
            "loc"             : loc,
            "scale"           : scale,
            "valid"           : valid,
            "invertible"      : invertible,
            "roundtrip_error" : roundtrip,
            "composite"       : composite if valid else 0.0,
            "scores"          : scores,
            **metrics,
        }

## Exhaustive evaluation

All 7 presets against all 12 slots. Fitting and scoring run on the pooled train values; nothing is cached
between combinations, so each cell of the grid is an independent fit exactly as `StatsComputer` would
produce it for that configuration.

In [ ]:
evaluator = StrategyEvaluator(study_config)

results = {}
for slot in ALL_SLOTS:
    for preset_name, strategy in StrategyCatalog.PRESETS.items():
        results[(slot, preset_name)] = evaluator.evaluate(slot, preset_name, strategy, pools[slot])

n_valid = sum(1 for entry in results.values() if entry["valid"])
print(f"evaluated {len(results)} combinations, {n_valid} valid, {len(results) - n_valid} gated out")

In [ ]:
class RankingRenderer:
    def __init__(self, results: dict) -> None:
        self.results = results

    def _slot_entries(self, slot: str) -> list:
        entries = [self.results[(slot, name)] for name in StrategyCatalog.PRESETS]

        return sorted(entries, key=lambda entry: entry["composite"], reverse=True)

    def render(self, slot: str) -> None:
        standard_name = StrategyCatalog.name_of(StrategyCatalog.standard_for(slot))

        rows = []
        for rank, entry in enumerate(self._slot_entries(slot), start=1):
            marker = " **(standard)**" if entry["strategy"] == standard_name else ""

            rows.append((
                rank,
                f"`{entry['strategy']}`{marker}",
                "yes" if entry["valid"] else "no",
                f"{entry['composite']:.3f}",
                f"{entry['std']:.3f}",
                f"{entry['mean']:+.3f}",
                f"{entry['skewness']:+.2f}",
                f"{entry['kurtosis']:+.1f}",
                f"{entry['tail_width']:.2f}",
                f"{100 * entry['outlier_fraction']:.3f}%",
                f"{entry['entropy']:.3f}",
                f"{entry['roundtrip_error']:.1e}",
            ))

        MarkdownRenderer.heading(slot)
        MarkdownRenderer.table(["Rank", "Strategy", "Valid", "Composite", "Std", "Mean", "Skew", "Kurt", "Tail width", "Outliers", "Entropy", "Round-trip"], rows)


ranking_renderer = RankingRenderer(results)
for slot in ALL_SLOTS:
    ranking_renderer.render(slot)

## Composite score across the full grid

The heatmap is the study at a glance: rows are slots, columns are strategies, cell values are composite
scores. Gated combinations (non-invertible or non-finite) sit at zero. The configured standard of each
slot is outlined in red.

> **What you should see:** zeroed log1p columns on every signed slot; `FIXED_DIV_PI` competitive only on
> the phase rows; a bright diagonal-ish band where the configured standards live if the current mapping
> is healthy.

In [ ]:
from matplotlib.patches import Rectangle


class GridHeatmap:
    def __init__(self, results: dict, slots: list, preset_names: list) -> None:
        self.results      = results
        self.slots        = slots
        self.preset_names = preset_names

    def _matrix(self, key: str) -> np.ndarray:
        return np.array([[self.results[(slot, name)][key] for name in self.preset_names] for slot in self.slots])

    def _decorate(self, axis) -> None:
        axis.set_xticks(range(len(self.preset_names)))
        axis.set_xticklabels(self.preset_names, rotation=40, ha="right")
        axis.set_yticks(range(len(self.slots)))
        axis.set_yticklabels(self.slots)

        for row, slot in enumerate(self.slots):
            column = self.preset_names.index(StrategyCatalog.name_of(StrategyCatalog.standard_for(slot)))
            axis.add_patch(Rectangle((column - 0.5, row - 0.5), 1, 1, fill=False, edgecolor="#c1121f", linewidth=2.0))

    def plot_composite(self) -> None:
        matrix = self._matrix("composite")

        figure, axis = plt.subplots(figsize=(9.5, 6.5))
        image = axis.imshow(matrix, cmap="viridis", vmin=0.0, vmax=1.0, aspect="auto")

        for row in range(matrix.shape[0]):
            for column in range(matrix.shape[1]):
                color = "white" if matrix[row, column] < 0.5 else "black"
                axis.text(column, row, f"{matrix[row, column]:.2f}", ha="center", va="center", color=color, fontsize=8)

        self._decorate(axis)
        axis.set_title("Composite score — slots vs strategies (standard outlined)")
        figure.colorbar(image, ax=axis, label="composite score")
        plt.show()
        plt.close(figure)

    def plot_roundtrip(self) -> None:
        matrix = np.log10(self._matrix("roundtrip_error") + 1e-16)

        figure, axis = plt.subplots(figsize=(9.5, 6.5))
        image = axis.imshow(matrix, cmap="magma_r", aspect="auto")

        for row in range(matrix.shape[0]):
            for column in range(matrix.shape[1]):
                axis.text(column, row, f"{matrix[row, column]:.0f}", ha="center", va="center", color="#2b2b2b", fontsize=8)

        self._decorate(axis)
        axis.set_title("Round-trip relative error, log10 — invertibility gate at -3")
        figure.colorbar(image, ax=axis, label="log10 relative error")
        plt.show()
        plt.close(figure)


heatmap = GridHeatmap(results, ALL_SLOTS, list(StrategyCatalog.PRESETS))
heatmap.plot_composite()

In [ ]:
heatmap.plot_roundtrip()

## Normalized distributions per slot

For each slot, the density of the normalized pool under every **valid** strategy, on a shared axis clipped
to $[-5, 5]$. Min-max variants occupy $[0, 1]$ by construction; centered variants spread around zero.
Densities are on a logarithmic axis so tail behaviour — the part that hurts training — stays visible.

In [ ]:
class OverlayPlotter:
    PALETTE = {
        "MIN_MAX"          : "#1f77b4",
        "MIN_MAX_LOG1P"    : "#aec7e8",
        "ROBUST_IQR"       : "#2ca02c",
        "ROBUST_IQR_LOG1P" : "#98df8a",
        "ZSCORE"           : "#d62728",
        "ZSCORE_LOG1P"     : "#ff9896",
        "FIXED_DIV_PI"     : "#7f7f7f",
    }

    def __init__(self, evaluator: StrategyEvaluator, results: dict, sampler: PlotSampler) -> None:
        self.evaluator = evaluator
        self.results   = results
        self.sampler   = sampler

    def plot(self, slot: str, values: np.ndarray) -> None:
        sample       = self.sampler.take(values)
        figure, axis = plt.subplots(figsize=(7.5, 3.2))

        for preset_name, strategy in StrategyCatalog.PRESETS.items():
            entry = self.results[(slot, preset_name)]
            if not entry["valid"]:
                continue

            normalized = self.evaluator.transform(sample, strategy, entry["loc"], entry["scale"])
            clipped    = np.clip(normalized, -5.0, 5.0)
            label      = f"{preset_name}  ({entry['composite']:.2f})"

            axis.hist(clipped, bins=160, range=(-5.0, 5.0), density=True, histtype="step", linewidth=1.2, color=self.PALETTE[preset_name], label=label)

        axis.set_yscale("log")
        axis.set_xlim(-5.0, 5.0)
        axis.set_title(f"Normalized distributions — {slot} (valid strategies, composite in legend)")
        axis.set_xlabel("normalized value")
        axis.set_ylabel("density (log)")
        axis.legend(loc="upper right", framealpha=0.9)
        plt.show()
        plt.close(figure)


overlay_plotter = OverlayPlotter(evaluator, results, plot_sampler)
for slot in ALL_SLOTS:
    overlay_plotter.plot(slot, pools[slot])

## Verdict — best strategy vs configured standard

In [ ]:
class VerdictRenderer:
    def __init__(self, results: dict) -> None:
        self.results = results

    def _ranked(self, slot: str) -> list:
        entries = [self.results[(slot, name)] for name in StrategyCatalog.PRESETS]

        return sorted(entries, key=lambda entry: entry["composite"], reverse=True)

    def render(self) -> None:
        rows       = []
        agreements = 0

        for slot in ALL_SLOTS:
            ranked        = self._ranked(slot)
            winner        = ranked[0]
            standard_name = StrategyCatalog.name_of(StrategyCatalog.standard_for(slot))
            standard      = self.results[(slot, standard_name)]
            standard_rank = 1 + [entry["strategy"] for entry in ranked].index(standard_name)
            agreement     = winner["strategy"] == standard_name
            agreements   += int(agreement)

            rows.append((
                f"`{slot}`",
                f"`{winner['strategy']}`",
                f"{winner['composite']:.3f}",
                f"`{standard_name}`",
                f"{standard['composite']:.3f}",
                standard_rank,
                "**agree**" if agreement else "differ",
            ))

        MarkdownRenderer.heading("Best strategy per slot vs configured standard", level=3)
        MarkdownRenderer.table(["Slot", "Best", "Best score", "Standard", "Standard score", "Standard rank", "Verdict"], rows)
        display(Markdown(f"**{agreements} of {len(ALL_SLOTS)} configured standards are the top-ranked strategy on this data.** Slots marked *differ* are candidates for a confirmation training sweep before changing `_SLOT_STRATEGIES`."))


VerdictRenderer(results).render()

In [ ]:
checks = [
    ("All 12 slot pools are non-empty",                       all(len(pools[slot]) > 0 for slot in ALL_SLOTS)),
    ("All pools are fully finite",                            all(np.isfinite(pools[slot]).all() for slot in ALL_SLOTS)),
    ("Every slot has at least one valid strategy",            all(any(results[(slot, name)]["valid"] for name in StrategyCatalog.PRESETS) for slot in ALL_SLOTS)),
    ("Configured standard is valid on every slot",            all(results[(slot, StrategyCatalog.name_of(StrategyCatalog.standard_for(slot)))]["valid"] for slot in ALL_SLOTS)),
    ("Phase pools bounded by pi",                             all(np.abs(pools[slot]).max() <= np.pi + 1e-6 for slot in ("pass/phase", "ifg/phase"))),
    ("Magnitude and amplitude pools are non-negative",        all(pools[slot].min() >= 0.0 for slot in ("pass/mag", "ifg/mag", "out/amp", "out/sigma"))),
]

for description, condition in checks:
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {description}")

print()

for slot in ALL_SLOTS:
    standard_name = StrategyCatalog.name_of(StrategyCatalog.standard_for(slot))
    ranked        = sorted(StrategyCatalog.PRESETS, key=lambda name: results[(slot, name)]["composite"], reverse=True)
    rank          = 1 + ranked.index(standard_name)
    status        = "PASS" if rank <= 3 else "FAIL"
    print(f"[{status}] {slot:<16} configured standard {standard_name} ranks {rank}/7")

### Common mistakes — interpreting this study

| Symptom | Likely cause | How to diagnose |
|---------|-------------|-----------------|
| log1p variants score 0 on signed slots (raw/norm re-im, phase, mu, DEM) | log1p clamps negatives to zero before the logarithm, so the transform is not invertible there | round-trip column is O(1) instead of below 1e-12 |
| `FIXED_DIV_PI` scores near 0 outside phase slots | division by $\pi$ is a physical bound for angles only; magnitudes overflow the unit range by orders of magnitude | tail-width and outlier columns explode for those cells |
| All strategies look similar on phase slots | phase is already bounded in $[-\pi, \pi]$, so the data-driven fits converge to similar scales | compare the fitted loc and scale across the phase ranking table |
| Fitted loc/scale differ from a training run's `normalization_stats.json` | this study fits on a clipped train crop with its own subsample cap; training fits on patched samples with `max_samples=4000` | compare against the JSON of the run, differences should be small but non-zero |
| Scores change between runs | unseeded subsampling | every random path in this notebook derives from `SEED = 42`; re-check after edits |
| A configured standard is not ranked first | the composite measures distribution shape for network consumption, not downstream accuracy | treat top-3 membership as the health bar; settle contested slots with a training sweep over the contenders |

**Expected visual outcome.** The composite heatmap shows zeroed log1p and `FIXED_DIV_PI` cells on signed
slots, with the red-outlined configured standards sitting in bright cells. Magnitude, amplitude and sigma
rows favour the log1p variants, which collapse their multi-decade tails; the overlay plots for those slots
show the log1p densities compact and centered while the linear fits leave a long right tail. Phase rows
score highest for `FIXED_DIV_PI` among physically meaningful choices, with near-identical bounded overlays.
The verdict table then states, per slot, whether the configured standard is already the best choice on this
data or which alternative to take into a confirmation training sweep.